# Solución P01 · Operaciones explícitas con pandas

Referencia de la clase 2: conserva cada paso de lectura, inspección y verificación, sin importar scripts ni funciones del curso. Cambiar la ruta de `repositorio` si la copia no está en la ubicación WSL indicada. Ejecutar celda por celda; las muestras no se descargan ni se modifican. [Enunciado](../talleres/P01.md).

## Preparar las muestras

Desde la terminal, con el entorno activo y situado en `kit/`:

```bash
python 00_datos.py --descargar pqrs
python 00_datos.py --verificar pqrs
```

El grupo `pqrs` obtiene las tres muestras de 1.000 filas en `kit/data/pqrs_muestras/`. Si ya vienen con tu clon y coinciden con el manifiesto, las conserva. No descarga los completos. Consulta [P01: actividad 0](../talleres/P01.md). Si solo tienes este notebook, necesitas también los scripts y manifiestos del repositorio. La descarga prepara archivos; a continuación tú lees, inspeccionas y comparas con pandas.

In [1]:
import pandas as pd

repositorio = "/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate"
archivo_2023_II = repositorio + "/kit/data/pqrs_muestras/Muestra_1000_2023_II.csv"
archivo_2024_I = repositorio + "/kit/data/pqrs_muestras/Muestra_1000_2024_I.csv"
archivo_2024_II = repositorio + "/kit/data/pqrs_muestras/Muestra_1000_2024_II.csv"
print(archivo_2023_II)
print(archivo_2024_I)
print(archivo_2024_II)
print("pandas:", pd.__version__)

/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/data/pqrs_muestras/Muestra_1000_2023_II.csv
/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/data/pqrs_muestras/Muestra_1000_2024_I.csv
/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/data/pqrs_muestras/Muestra_1000_2024_II.csv
pandas: 2.2.3


## 1. Mirar el archivo antes de leerlo
`open` permite inspeccionar la cabecera. El manifiesto describe la procedencia, los separadores y el esquema esperado. `json.load` solo lee ese documento, no transforma los CSV.

In [2]:
with open(archivo_2023_II, encoding="utf-8-sig") as archivo:
    print(archivo.readline())
with open(archivo_2024_I, encoding="utf-8-sig") as archivo:
    print(archivo.readline())
with open(archivo_2024_II, encoding="utf-8-sig") as archivo:
    print(archivo.readline())

OBJECTID;periodo;mes;pqr_canal;pet_cod_depto;pet_cod_mpio;id_afec;afec_parentesco;afec_genero;afec_edad;afec_edadr;afec_educ;afec_regafiliacion;afec_getnico;afec_pobespecial;afec_cod_depto;afec_cod_mpio;ent_nombre;ent_tipovig_sns;ent_cod_sns;ent_alias_sn;ent_cod_depto;ent_cod_mpio;cod_macromot;macromotivo;cod_motgen;motivo_general;cod_motesp;motivo_especifico;cod_tipo_motesp;tipo_de_motivo_especifico;cod_subtipo_motesp;subtipo_de_motivo_especifico;patologia_1;patologia_tipo;cie_10;alto_costo;clasificacion_de_riesgo

OBJECTID,periodo,mes,pqr_canal,pet_cod_depto,pet_cod_mpio,id_afec,afec_parentesco,afec_genero,afec_edad,afec_edadr,afec_educ,afec_regafiliacion,afec_getnico,afec_pobespecial,afec_cod_depto,afec_cod_mpio,ent_nombre,ent_tipovig_sns,ent_cod_sns,ent_alias_sn,ent_cod_depto,ent_cod_mpio,cod_macromot,macromotivo,cod_motgen,motivo_general,cod_motesp,motivo_especifico,cod_tipo_motesp,tipo_de_motivo_especifico,cod_subtipo_motesp,subtipo_de_motivo_especifico,patologia_1,patologia_tipo

In [3]:
import json
with open(repositorio + "/kit/pqrs_fuentes.json", encoding="utf-8") as archivo:
    manifiesto = json.load(archivo)
df_manifiesto = pd.DataFrame(manifiesto).set_index("periodo")
display(df_manifiesto[["archivo", "separador", "sha256_muestra"]])

,archivo,separador,sha256_muestra
periodo,,,
2023_II,PQRD_2023_II.csv,;,5546bb62ca598654a821c147d1efdc92f165324c13f904...
2024_I,PQRD_2024_I.csv,",",f06abcc52537eff223894edbeb83aedd9719d45b7d0887...
2024_II,PQRD_2024_II.csv,;,38102044d2c66b86b1e081867bbc854c4ea220524b51a0...


**Antes de continuar:** identifica qué archivo usa coma y cuáles usan punto y coma. Predice qué pasaría si usaras el separador incorrecto.

## 2. Leer cada CSV explícitamente
`sep` indica el separador; `dtype=str` conserva códigos y ceros iniciales; `keep_default_na=False` conserva textos vacíos y marcadores en lugar de interpretarlos automáticamente como nulos. `encoding="utf-8-sig"` admite la marca BOM.

In [4]:
df_2023_II = pd.read_csv(
    archivo_2023_II,
    sep=";",
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False
)
display(df_2023_II.head())
print(df_2023_II.shape)

,OBJECTID,periodo,mes,pqr_canal,pet_cod_depto,pet_cod_mpio,id_afec,afec_parentesco,afec_genero,afec_edad,...,motivo_especifico,cod_tipo_motesp,tipo_de_motivo_especifico,cod_subtipo_motesp,subtipo_de_motivo_especifico,patologia_1,patologia_tipo,cie_10,alto_costo,clasificacion_de_riesgo
0,1,2023,7,Personalizado,11,11001,US- 000002456900,A nombre propio,HOMBRE,69,...,FALTA DE OPORTUNIDAD EN LA ATENCIÓN EN OTROS S...,01010308,PROCEDIMIENTOS NO QUIRÚRGICOS,01010308001,IMAGENOLOGÍA,ENFERMEDADES CRÓNICAS NO TRASMISIBLES CARDIOVA...,HTA (HIPERTENSIÓN ARTERIAL),"HERNIA ABDOMINAL NO ESPECIFICADA, SIN OBSTRUCC...",No aplica,SIMPLE
1,2,2023,7,Web,11,11001,US- 000002368140,A nombre propio,MUJER,35,...,FALLAS O IRREGULARIDADES EN EL MANEJO TÉCNICO ...,,,,,ENFERMEDADES HUÉRFANAS,OTROS,"ENFERMEDAD DE CROHN, NO ESPECIFICADA",No aplica,PRIORIZADO
2,3,2023,7,Personalizado,70,70702,US- 000003298634,A nombre propio,MUJER,77,...,FALTA DE OPORTUNIDAD EN LA AUTORIZACIÓN DE TEC...,01030101,MEDICAMENTOS UPC,,,ENFERMEDADES CRÓNICAS NO TRASMISIBLES CARDIOVA...,HTA (HIPERTENSIÓN ARTERIAL),DIABETES MELLITUS INSULINODEPENDIENTE,No aplica,PRIORIZADO
3,4,2023,7,Escrito,76,76892,US- 000003300829,Otro,HOMBRE,71,...,FALTA DE OPORTUNIDAD EN LA ENTREGA O ENTREGA I...,01010401,MEDICAMENTOS UPC,,,PROBLEMAS RELACIONADOS CON FACILIDADES DE ATEN...,PROBLEMAS RELACIONADOS CON FACILIDADES DE ATEN...,PROBLEMAS RELACIONADOS CON FACILIDADES DE ATEN...,No aplica,SIMPLE
4,5,2023,7,Personalizado,08,08001,US- 000003286661,A nombre propio,MUJER,48,...,FALTA DE OPORTUNIDAD EN LA AUTORIZACIÓN DE OTR...,01030409,PROCEDIMIENTOS QUIRÚRGICOS,01030409003,OTROS PROCEDIMIENTOS QUIRÚRGICOS,PROBLEMAS RELACIONADOS CON FACILIDADES DE ATEN...,PROBLEMAS RELACIONADOS CON FACILIDADES DE ATEN...,"HEMORRAGIA VAGINAL Y UTERINA ANORMAL, NO ESPEC...",No aplica,SIMPLE


(1000, 38)


In [5]:
df_2024_I = pd.read_csv(
    archivo_2024_I,
    sep=",",
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False
)
display(df_2024_I.head())
print(df_2024_I.shape)

,OBJECTID,periodo,mes,pqr_canal,pet_cod_depto,pet_cod_mpio,id_afec,afec_parentesco,afec_genero,afec_edad,...,motivo_especifico,cod_tipo_motesp,tipo_de_motivo_especifico,cod_subtipo_motesp,subtipo_de_motivo_especifico,patologia_1,patologia_tipo,cie_10,alto_costo,clasificacion_de_riesgo
0,1,2024,6,Telefónico,5,5001,US- 000003606236,Hijo (A),HOMBRE,73,...,FALTA DE OPORTUNIDAD EN LA AUTORIZACIÓN DE OTR...,1030408,PROCEDIMIENTOS NO QUIRÚRGICOS,1030408001,IMAGENOLOGÍA,ENFERMEDADES OSTEOARTICULARES,OTROS,SÍNDROME DE ABDUCCIÓN DOLOROSA DEL HOMBRO,No aplica,SIMPLE
1,2,2024,4,Redes Sociales,63,63401,US- 000003606237,A nombre propio,HOMBRE,84,...,NEGACIÓN PARA LA ENTREGA DE TECNOLOGÍAS EN SAL...,1020402,MEDICAMENTOS NO UPC,,,PROBLEMAS RELACIONADOS CON FACILIDADES DE ATEN...,PROBLEMAS RELACIONADOS CON FACILIDADES DE ATEN...,OTROS TRASTORNOS DEL SISTEMA URINARIO,No aplica,PRIORIZADO
2,3,2024,5,Web,11,11001,US- 000003606238,A nombre propio,HOMBRE,0,...,FALTA DE OPORTUNIDAD EN LA ATENCIÓN EN OTROS S...,1010308,PROCEDIMIENTOS NO QUIRÚRGICOS,1010308001,IMAGENOLOGÍA,MATERNO INFANTIL,ATENCIÓN NIÑOS MENORES DE 5 AÑOS,CONTROL DE SALUD DE RUTINA DEL NIÑO,No aplica,SIMPLE
3,4,2024,6,Web,11,11001,US- 000001592546,Cónyuge,MUJER,62,...,FALTA DE OPORTUNIDAD EN LA AUTORIZACIÓN DE TEC...,1030101,MEDICAMENTOS UPC,,,ENFERMEDADES OSTEOARTICULARES,OTROS,"ARTRITIS REUMATOIDE SEROPOSITIVA, SIN OTRA ESP...",No aplica,SIMPLE
4,5,2024,6,Personalizado,66,66001,US- 000003606239,A nombre propio,MUJER,39,...,FALTA DE OPORTUNIDAD EN LA ATENCIÓN EN OTROS S...,1010308,PROCEDIMIENTOS NO QUIRÚRGICOS,1010308003,QUIMIOTERAPIA,CÁNCER,CÁNCER DE ÚTERO,TUMOR MALIGNO DEL FONDO DEL ÚTERO,Quimioterapia y radioterapia para el cáncer,PRIORIZADO


(1000, 38)


In [6]:
df_2024_II = pd.read_csv(
    archivo_2024_II,
    sep=";",
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False
)
display(df_2024_II.head())
print(df_2024_II.shape)

,OBJECTID,periodo,mes,pqr_canal,pet_cod_depto,pet_cod_mpio,id_afec,afec_parentesco,afec_genero,afec_edad,...,motivo_especifico,cod_tipo_motesp,tipo_de_motivo_especifico,cod_subtipo_motesp,subtipo_de_motivo_especifico,patologia_1,patologia_tipo,cie_10,alto_costo,clasificacion_de_riesgo
0,1,2024,7,Web,05,05789,US- 000002469551,A nombre propio,MUJER,71,...,FALTA DE OPORTUNIDAD EN LAS CITAS O CONSULTAS,01010103,ESPECIALIDADES MÉDICAS,01010103010,CIRUGÍA CARDIOVASCULAR,ENFERMEDADES CRÓNICAS NO TRASMISIBLES CARDIOVA...,OTROS,PROLAPSO (DE LA VÁLVULA) MITRAL,No aplica,PRIORIZADO
1,2,2024,7,Web,11,11001,US- 000004066349,A nombre propio,HOMBRE,61,...,NEGACIÓN EN LA ATENCIÓN EN OTROS SERVICIOS DE ...,01020308,PROCEDIMIENTOS NO QUIRÚRGICOS,01020308001,IMAGENOLOGÍA,PROBLEMAS RELACIONADOS CON FACILIDADES DE ATEN...,PROBLEMAS RELACIONADOS CON FACILIDADES DE ATEN...,GLAUCOMA PRIMARIO DE ANGULO CERRADO,No aplica,SIMPLE
2,3,2024,7,Web,66,66001,US- 000000454297,A nombre propio,MUJER,32,...,FALTA DE OPORTUNIDAD EN LAS CITAS O CONSULTAS,01010103,ESPECIALIDADES MÉDICAS,01010103033,DERMATOLOGÍA,ENFERMEDADES CRÓNICAS NO TRANSMISIBLES –DIABETES,DIABETES,DIABETES MELLITUS NO INSULINODEPENDIENTE,No aplica,SIMPLE
3,4,2024,7,Web,66,66001,US- 000000455893,Madre,HOMBRE,9,...,FALTA DE OPORTUNIDAD EN LAS CITAS O CONSULTAS,01010103,ESPECIALIDADES MÉDICAS,01010103080,OTORRINOLARINGOLOGÍA,PROBLEMAS RELACIONADOS CON FACILIDADES DE ATEN...,PROBLEMAS RELACIONADOS CON FACILIDADES DE ATEN...,APNEA DEL SUEÑO,No aplica,SIMPLE
4,5,2024,7,Telefónico,05,05664,US- 000004066350,Hermano (A),HOMBRE,33,...,FALTA DE OPORTUNIDAD EN LA ATENCIÓN EN OTROS S...,01010301,URGENCIAS,,,PROBLEMAS RELACIONADOS CON FACILIDADES DE ATEN...,PROBLEMAS RELACIONADOS CON FACILIDADES DE ATEN...,"FIEBRE RECURRENTE, NO ESPECIFICADA",No aplica,PRIORIZADO


(1000, 38)


## 3. Inspeccionar y comparar los esquemas
Cada muestra debe tener 1.000 filas y 38 columnas. Primero vemos los resultados; después escribimos `assert` para detenernos si no se cumple el contrato. No basta con contar columnas: también comparamos nombres y orden.

In [7]:
df_2023_II.info()
display(df_2023_II.dtypes)
print(df_2023_II.columns.tolist())
print("2023 frente a 2024-I:", df_2023_II.columns.equals(df_2024_I.columns))
print("2023 frente a 2024-II:", df_2023_II.columns.equals(df_2024_II.columns))
assert df_2023_II.shape == (1000, 38)
assert df_2024_I.shape == (1000, 38)
assert df_2024_II.shape == (1000, 38)
assert df_2023_II.columns.tolist() == df_manifiesto.loc["2023_II", "columnas"]
assert df_2024_I.columns.tolist() == df_manifiesto.loc["2024_I", "columnas"]
assert df_2024_II.columns.tolist() == df_manifiesto.loc["2024_II", "columnas"]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 38 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   OBJECTID                      1000 non-null   object
 1   periodo                       1000 non-null   object
 2   mes                           1000 non-null   object
 3   pqr_canal                     1000 non-null   object
 4   pet_cod_depto                 1000 non-null   object
 5   pet_cod_mpio                  1000 non-null   object
 6   id_afec                       1000 non-null   object
 7   afec_parentesco               1000 non-null   object
 8   afec_genero                   1000 non-null   object
 9   afec_edad                     1000 non-null   object
 10  afec_edadr                    1000 non-null   object
 11  afec_educ                     1000 non-null   object
 12  afec_regafiliacion            1000 non-null   object
 13  afec_getnico       

OBJECTID                        object
periodo                         object
mes                             object
pqr_canal                       object
pet_cod_depto                   object
pet_cod_mpio                    object
id_afec                         object
afec_parentesco                 object
afec_genero                     object
afec_edad                       object
afec_edadr                      object
afec_educ                       object
afec_regafiliacion              object
afec_getnico                    object
afec_pobespecial                object
afec_cod_depto                  object
afec_cod_mpio                   object
ent_nombre                      object
ent_tipovig_sns                 object
ent_cod_sns                     object
ent_alias_sn                    object
ent_cod_depto                   object
ent_cod_mpio                    object
cod_macromot                    object
macromotivo                     object
cod_motgen               

['OBJECTID', 'periodo', 'mes', 'pqr_canal', 'pet_cod_depto', 'pet_cod_mpio', 'id_afec', 'afec_parentesco', 'afec_genero', 'afec_edad', 'afec_edadr', 'afec_educ', 'afec_regafiliacion', 'afec_getnico', 'afec_pobespecial', 'afec_cod_depto', 'afec_cod_mpio', 'ent_nombre', 'ent_tipovig_sns', 'ent_cod_sns', 'ent_alias_sn', 'ent_cod_depto', 'ent_cod_mpio', 'cod_macromot', 'macromotivo', 'cod_motgen', 'motivo_general', 'cod_motesp', 'motivo_especifico', 'cod_tipo_motesp', 'tipo_de_motivo_especifico', 'cod_subtipo_motesp', 'subtipo_de_motivo_especifico', 'patologia_1', 'patologia_tipo', 'cie_10', 'alto_costo', 'clasificacion_de_riesgo']
2023 frente a 2024-I: True
2023 frente a 2024-II: True


## 4. Seleccionar campos y reconocer faltantes
Prueba primero sobre un campo; luego cambia el nombre por otro que quieras investigar. Vacío, nulo y `ND` se registran por separado. No reemplazamos ninguno por cero.

In [8]:
campo = "pet_cod_mpio"
valores = df_2023_II[campo].str.strip()
print("Nulos:", valores.isna().sum())
print("Textos vacíos:", valores.eq("").sum())
print("Marcadores ND:", valores.str.upper().eq("ND").sum())
display(valores.value_counts(dropna=False).head(10))
display(df_2023_II.loc[valores.eq(""), ["OBJECTID", "pet_cod_mpio", "afec_cod_mpio"]])

Nulos: 0
Textos vacíos: 2
Marcadores ND: 0


pet_cod_mpio
11001    244
05001     89
76001     61
08001     38
66001     22
25754     19
05088     17
68001     17
73001     16
54001     15
Name: count, dtype: int64

,OBJECTID,pet_cod_mpio,afec_cod_mpio
717,718,,20228
953,954,,44078


## 5. Unir muestras conservando su procedencia
`assign` crea una copia con la columna del corte; `concat` apila filas. `periodo` es el año declarado en la fila, mientras que `corte_archivo` identifica de qué publicación vino. No confundir ambos.

In [9]:
df_pqrs = pd.concat([
    df_2023_II.assign(corte_archivo="2023_II"),
    df_2024_I.assign(corte_archivo="2024_I"),
    df_2024_II.assign(corte_archivo="2024_II")
], ignore_index=True)
print(df_pqrs.shape)
assert len(df_pqrs) == 3000
display(df_pqrs.groupby(["corte_archivo", "periodo", "mes"]).size().rename("filas").reset_index())

(3000, 39)


,corte_archivo,periodo,mes,filas
0,2023_II,2023,7,1000
1,2024_I,2024,1,11
2,2024_I,2024,2,9
3,2024_I,2024,3,17
4,2024_I,2024,4,47
5,2024_I,2024,5,188
6,2024_I,2024,6,728
7,2024_II,2024,7,1000


In [10]:
campos = ["periodo", "mes", "pet_cod_mpio", "afec_cod_mpio", "OBJECTID"]
perfil = pd.DataFrame(index=campos)
perfil["nulos"] = df_pqrs[campos].isna().sum()
# El bucle repite operaciones que ya observamos sobre una sola columna.
for campo in campos:
    valores = df_pqrs[campo].str.strip()
    perfil.loc[campo, "vacios"] = valores.eq("").sum()
    perfil.loc[campo, "ND"] = valores.str.upper().eq("ND").sum()
display(perfil)
es_repetido = df_pqrs.duplicated(["corte_archivo", "OBJECTID"], keep=False)
print("Filas con clave candidata repetida:", es_repetido.sum())
display(df_pqrs.loc[es_repetido, ["corte_archivo", "OBJECTID"]])

,nulos,vacios,ND
periodo,0,0.0,0.0
mes,0,0.0,0.0
pet_cod_mpio,0,3.0,0.0
afec_cod_mpio,0,0.0,0.0
OBJECTID,0,0.0,0.0


Filas con clave candidata repetida: 0


,corte_archivo,OBJECTID


**Interpretación:** peticionario y afectado pueden tener ubicaciones distintas. `id_afec` no es una clave de reclamo; corte–OBJECTID es una clave candidata, no una prueba de identidad semántica. Describe lo que observas antes de proponer una corrección.

## 6. Detectar valores no convertibles sin perderlos
La lectura como texto permite inspeccionar los valores. Creamos una serie numérica separada y distinguimos un vacío de un texto no convertible.

In [11]:
mes_texto = df_pqrs["mes"].str.strip()
mes_numero = pd.to_numeric(mes_texto, errors="coerce")
es_vacio = mes_texto.eq("") | mes_texto.isna()
no_convertible = mes_numero.isna() & ~es_vacio
fuera_rango = mes_numero.notna() & (~mes_numero.between(1, 12) | mes_numero.mod(1).ne(0))
df_revision_mes = df_pqrs.loc[es_vacio | no_convertible | fuera_rango, ["corte_archivo", "OBJECTID", "mes"]].copy()
print("Vacíos:", es_vacio.sum(), "No convertibles:", no_convertible.sum(), "Fuera de dominio:", fuera_rango.sum())
display(df_revision_mes)
# df_pqrs conserva mes como texto; no se descartaron filas.

Vacíos: 0 No convertibles: 0 Fuera de dominio: 0


,corte_archivo,OBJECTID,mes


## 7. Leer por bloques: observar antes de repetir
`chunksize` devuelve un lector de bloques. `next` toma solamente el primero. Modifica `tamano_bloque` y vuelve a ejecutar esta celda para ver qué cambia. Los bytes del DataFrame no son la memoria total del proceso (RSS).

In [12]:
tamano_bloque = 100
with pd.read_csv(archivo_2023_II, sep=";", encoding="utf-8-sig", dtype=str,
                 keep_default_na=False, chunksize=tamano_bloque) as lector:
    primer_bloque = next(lector)
display(primer_bloque.head())
print("Filas del bloque:", len(primer_bloque))
print("Bytes del DataFrame:", primer_bloque.memory_usage(deep=True).sum())

,OBJECTID,periodo,mes,pqr_canal,pet_cod_depto,pet_cod_mpio,id_afec,afec_parentesco,afec_genero,afec_edad,...,motivo_especifico,cod_tipo_motesp,tipo_de_motivo_especifico,cod_subtipo_motesp,subtipo_de_motivo_especifico,patologia_1,patologia_tipo,cie_10,alto_costo,clasificacion_de_riesgo
0,1,2023,7,Personalizado,11,11001,US- 000002456900,A nombre propio,HOMBRE,69,...,FALTA DE OPORTUNIDAD EN LA ATENCIÓN EN OTROS S...,01010308,PROCEDIMIENTOS NO QUIRÚRGICOS,01010308001,IMAGENOLOGÍA,ENFERMEDADES CRÓNICAS NO TRASMISIBLES CARDIOVA...,HTA (HIPERTENSIÓN ARTERIAL),"HERNIA ABDOMINAL NO ESPECIFICADA, SIN OBSTRUCC...",No aplica,SIMPLE
1,2,2023,7,Web,11,11001,US- 000002368140,A nombre propio,MUJER,35,...,FALLAS O IRREGULARIDADES EN EL MANEJO TÉCNICO ...,,,,,ENFERMEDADES HUÉRFANAS,OTROS,"ENFERMEDAD DE CROHN, NO ESPECIFICADA",No aplica,PRIORIZADO
2,3,2023,7,Personalizado,70,70702,US- 000003298634,A nombre propio,MUJER,77,...,FALTA DE OPORTUNIDAD EN LA AUTORIZACIÓN DE TEC...,01030101,MEDICAMENTOS UPC,,,ENFERMEDADES CRÓNICAS NO TRASMISIBLES CARDIOVA...,HTA (HIPERTENSIÓN ARTERIAL),DIABETES MELLITUS INSULINODEPENDIENTE,No aplica,PRIORIZADO
3,4,2023,7,Escrito,76,76892,US- 000003300829,Otro,HOMBRE,71,...,FALTA DE OPORTUNIDAD EN LA ENTREGA O ENTREGA I...,01010401,MEDICAMENTOS UPC,,,PROBLEMAS RELACIONADOS CON FACILIDADES DE ATEN...,PROBLEMAS RELACIONADOS CON FACILIDADES DE ATEN...,PROBLEMAS RELACIONADOS CON FACILIDADES DE ATEN...,No aplica,SIMPLE
4,5,2023,7,Personalizado,08,08001,US- 000003286661,A nombre propio,MUJER,48,...,FALTA DE OPORTUNIDAD EN LA AUTORIZACIÓN DE OTR...,01030409,PROCEDIMIENTOS QUIRÚRGICOS,01030409003,OTROS PROCEDIMIENTOS QUIRÚRGICOS,PROBLEMAS RELACIONADOS CON FACILIDADES DE ATEN...,PROBLEMAS RELACIONADOS CON FACILIDADES DE ATEN...,"HEMORRAGIA VAGINAL Y UTERINA ANORMAL, NO ESPEC...",No aplica,SIMPLE


Filas del bloque: 100
Bytes del DataFrame: 268976


Ahora repetimos con 100, 500 y 1.000 filas sobre **los tres archivos**. El alumno puede modificar tamaños y observar cada bloque. El bucle es visible: suma filas, acumula vacíos y guarda la memoria máxima observada. Cada combinación se compara con la lectura completa, no solo con otra ejecución por bloques.

In [13]:
mediciones = []
archivos = [archivo_2023_II, archivo_2024_I, archivo_2024_II]
separadores = [";", ",", ";"]
dataframes = [df_2023_II, df_2024_I, df_2024_II]
cortes = ["2023_II", "2024_I", "2024_II"]

for posicion in range(3):
    for tamano_bloque in [100, 500, 1000]:
        total_filas = 0
        total_vacios = pd.Series(0, index=dataframes[posicion].columns)
        memoria_maxima = 0
        with pd.read_csv(archivos[posicion], sep=separadores[posicion], encoding="utf-8-sig",
                         dtype=str, keep_default_na=False, chunksize=tamano_bloque) as lector:
            for bloque in lector:
                total_filas += len(bloque)
                memoria_bloque = int(bloque.memory_usage(deep=True).sum())
                memoria_maxima = max(memoria_maxima, memoria_bloque)
                for campo in bloque.columns:
                    total_vacios[campo] += bloque[campo].str.strip().eq("").sum()
        assert total_filas == len(dataframes[posicion])
        for campo in total_vacios.index:
            assert total_vacios[campo] == dataframes[posicion][campo].str.strip().eq("").sum()
        mediciones.append({"corte": cortes[posicion], "bloque": tamano_bloque,
                           "filas": total_filas, "memoria_dataframe_max_bytes": memoria_maxima})

comparacion_bloques = pd.DataFrame(mediciones)
display(comparacion_bloques)

,corte,bloque,filas,memoria_dataframe_max_bytes
0,2023_II,100,1000,271361
1,2023_II,500,1000,1346842
2,2023_II,1000,1000,2693128
3,2024_I,100,1000,269230
4,2024_I,500,1000,1342253
5,2024_I,1000,1000,2682481
6,2024_II,100,1000,271340
7,2024_II,500,1000,1345684
8,2024_II,1000,1000,2688807


## 8. Verificar el archivo: bytes y SHA-256
Pandas verifica datos; SHA-256 verifica bytes del archivo. Para estas muestras pequeñas leemos cada archivo en memoria con `read()`. Para los completos, más adelante se implementará lectura binaria por bloques. El hash de la muestra se compara con `sha256_muestra`, no con el de su archivo completo.

In [14]:
import hashlib

huellas = []
for posicion in range(3):
    with open(archivos[posicion], "rb") as archivo:
        contenido = archivo.read()
    calculado = hashlib.sha256(contenido).hexdigest()
    esperado = df_manifiesto.loc[cortes[posicion], "sha256_muestra"]
    huellas.append({"corte": cortes[posicion], "bytes_archivo": len(contenido),
                    "sha256_calculado": calculado, "coincide": calculado == esperado})
    assert calculado == esperado, "Revisar el origen del archivo antes de continuar"
df_huellas = pd.DataFrame(huellas)
display(df_huellas)

,corte,bytes_archivo,sha256_calculado,coincide
0,2023_II,752214,5546bb62ca598654a821c147d1efdc92f165324c13f904...,True
1,2024_I,737025,f06abcc52537eff223894edbeb83aedd9719d45b7d0887...,True
2,2024_II,745755,38102044d2c66b86b1e081867bbc854c4ea220524b51a0...,True


## 9. Guardar una evidencia elegida por el alumno
Comprueba la ruta impresa antes de escribir. Estas instrucciones guardan únicamente tablas de control; repetirlas reemplaza esos CSV. Cambia el nombre si quieres conservar otra ejecución.

In [15]:
import os
carpeta_salida = repositorio + "/kit/salidas/clase2/P01"
print("Guardaré en:", carpeta_salida)
os.makedirs(carpeta_salida, exist_ok=True)
perfil.to_csv(carpeta_salida + "/perfil_campos.csv", index_label="campo")
comparacion_bloques.to_csv(carpeta_salida + "/comparacion_bloques.csv", index=False)
df_huellas.to_csv(carpeta_salida + "/verificacion_archivos.csv", index=False)
df_revision_mes.to_csv(carpeta_salida + "/revision_mes.csv", index=False)

Guardaré en: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/clase2/P01


## Entrega e interpretación propia

1. Explica separadores, esquema y tipos de los tres archivos.
2. Escribe un diccionario de cinco campos y distingue peticionario, afectado y reporte.
3. Compara memoria y conteos por tamaño de bloque; no extrapoles RSS a partir del CSV.
4. Justifica cómo tratarías faltantes y valores no convertibles sin borrar evidencia.
5. Declara dos límites: primeras 1.000 filas no aleatorias y reportes que no equivalen a prevalencia clínica.

Para archivos completos se verificarían también hashes, esquema, filas y anchos de registro con lectura por bloques. Las funciones que automatizan estos pasos se estudiarán después de la clase 2.

## Respuestas de referencia

Cada muestra contiene 1.000 filas y 38 columnas; la unión conserva 3.000 filas y añade una columna de procedencia. Los tres hashes coinciden con el manifiesto del corte. Cambiar el bloque conserva filas y vacíos; cambia la memoria del bloque, que no representa la RSS del proceso. En este corte no hay claves corte–OBJECTID repetidas ni meses en revisión.

| Campo | Interpretación |
|---|---|
| periodo | Año declarado en la fila |
| mes | Mes declarado; entero entre 1 y 12 |
| pet_cod_mpio | Ubicación municipal del peticionario |
| afec_cod_mpio | Ubicación municipal del afectado |
| OBJECTID | Identificador candidato dentro del corte |

Un faltante territorial no prueba anonimato. Los vacíos y marcadores deben conservarse hasta acordar reglas. La publicación exige revisar procedencia, esquema y conteos; las muestras no permiten estimar porcentajes nacionales ni necesidades de RAM para los completos.